## Week 7 — RAG-Augmented Few-Shot Price Estimation (mmaitsimwale)

### The capstone progression so far

- **Week 5 (RAG):** retrieve relevant knowledge-base chunks by embedding similarity, evaluate with LLM-as-judge, self-improve iteratively.
- **Week 6 (Zero-Shot Benchmark):** prompt frontier LLMs (OpenRouter + Groq) with a product description and ask for a price; no examples, no training.
- **Week 7 (Fine-Tuning):** QLoRA fine-tune Llama-3.2-3B on 20 k price-labelled examples so the model internalises pricing knowledge into its weights.

### The creative exercise: RAG-Augmented Few-Shot Pricing

**Insight:** A human expert estimating product prices doesn't guess in isolation, they recall
similar items they've seen before.  We can simulate this with retrieval.

At inference time:
1. Embed the test product's description with `all-MiniLM-L6-v2`.
2. Find the k most similar products in a training-set index (cosine similarity).
3. Inject those k analogies **with their known prices** as few-shot examples in the prompt.
4. Let the LLM reason: *"This item is similar to X ($49), Y ($62), Z ($55) → probably ~$55."*

This bridges Week 5 (retrieval) and Week 7 (training data matters): it shows how much
in-context analogies improve zero-shot accuracy, and quantifies what fine-tuning
still adds on top.  **No GPU required.**


## Architecture

```
Test Item (item.summary)
        │
        ▼
 Embed (all-MiniLM-L6-v2)            Training Index
        │                           (2 000 items, pre-embedded)
        └──── Cosine Similarity ──────────────────┘
                        │
                        ▼
              Top-k Similar Items + Prices
                        │
              ┌─────────┴──────────┐
              │  Few-Shot Prompt   │   Zero-Shot Prompt
              │  (analogies +      │   (description only)
              │   description)     │
              └─────────┬──────────┘
                        │
               Groq / OpenRouter LLM
                        │
                   Price Estimate
                        │
                pricer.evaluator.Tester
          (avg absolute error $ + R²)
```

**Experiment (What I want to know is):** does injecting k similar examples reduce average price-estimation error
compared with zero-shot?  I test k = 3 and k = 5 on two providers.


In [1]:
# Setup: imports, repo-root resolution, pricer module path
import os
import re
import sys
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import r2_score
from tqdm import tqdm


def _repo_root() -> Path:
    """Walk up from cwd until week6/pricer is found."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "week6" / "pricer").is_dir():
            return cand
    raise RuntimeError(
        "Cannot find week6/pricer — start Jupyter from the llm_engineering repo root."
    )


REPO_ROOT = _repo_root()

# Reuse the Week 6 pricer module (Item + Tester)
WEEK6_PATH = str(REPO_ROOT / "week6")
if WEEK6_PATH not in sys.path:
    sys.path.insert(0, WEEK6_PATH)

from pricer.items import Item
from pricer.evaluator import Tester

BENCHMARK_SIZE = 50    # items per pricer — raise to 200 for more reliable stats
FEW_SHOT_K = [3, 5]   # k values to benchmark
INDEX_SIZE = 2000      # training items to embed and index

print(f"Repo root  : {REPO_ROOT}")
print(f"Week6 path : {WEEK6_PATH}")
print(f"Benchmark  : {BENCHMARK_SIZE} items/pricer, index size {INDEX_SIZE}")


Repo root  : /home/ed/work/projects/andela-ai-eng/llm_engineering
Week6 path : /home/ed/work/projects/andela-ai-eng/llm_engineering/week6
Benchmark  : 50 items/pricer, index size 2000


In [ ]:
# .env / key check
load_dotenv(REPO_ROOT / ".env", override=True)

hf_token        = os.getenv("HF_TOKEN")
or_api_key      = os.getenv("OR_API_KEY")
or_client_url   = os.getenv("OR_CLIENT_URL", "https://openrouter.ai/api/v1")
groq_api_key    = os.getenv("GROQ_API_KEY")
groq_client_url = os.getenv("GROQ_CLIENT_URL", "https://api.groq.com/openai/v1")

for name, val in [
    ("HF_TOKEN",     hf_token),
    ("OR_API_KEY",   or_api_key),
    ("GROQ_API_KEY", groq_api_key),
]:
    if val:
        print(f"{name} exists and begins {val[:8]}")
    else:
        print(f"{name} NOT SET")


HF_TOKEN exists and begins hf_BjcEY
OR_API_KEY exists and begins sk-or-v1
GROQ_API_KEY exists and begins gsk_dfVg


In [ ]:
# Two OpenAI-compatible clients

or_client = OpenAI(
    api_key=or_api_key,
    base_url=or_client_url,
)

groq_client = OpenAI(
    api_key=groq_api_key,
    base_url=groq_client_url,
)

print("OpenRouter:", or_client.base_url)
print("Groq      :", groq_client.base_url)


OpenRouter: https://openrouter.ai/api/v1/
Groq      : https://api.groq.com/openai/v1/


In [4]:
# Load Amazon product dataset — reuse ed-donner's curated items_lite
# ( dataset from Week 6; train set becomes our retrieval index)

from huggingface_hub import login

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("Logged in to HuggingFace")

train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"Loaded  {len(train):,} train | {len(val):,} val | {len(test):,} test")
print(f"Example : {test[0]}")
print(f"Summary : {test[0].summary[:100]}...")


Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
Logged in to HuggingFace


README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded  20,000 train | 1,000 val | 1,000 test
Example : title='Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal' category='Musical_Instruments' price=219.0 full=None weight=2.0 summary='Title: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.' prompt=None id=None
Summary : Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Des...


In [5]:
# Build the embedding index over INDEX_SIZE training items
# Using all-MiniLM-L6-v2 (Week 5 Day 2 model) — free, runs on CPU, ~5 s for 2000 items.

from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Loading sentence-transformer: {EMBED_MODEL_NAME}")
embedder = SentenceTransformer(EMBED_MODEL_NAME)

index_items = train[:INDEX_SIZE]
index_texts  = [item.summary for item in index_items]

print(f"Embedding {len(index_texts):,} training items...")
raw_matrix = embedder.encode(
    index_texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64,
)

# L2-normalise each row so dot product == cosine similarity
norms = np.linalg.norm(raw_matrix, axis=1, keepdims=True)
index_matrix = raw_matrix / (norms + 1e-9)

print(f"Index ready: {index_matrix.shape}  (rows=items, cols=embedding dim)")


Loading sentence-transformer: all-MiniLM-L6-v2
Embedding 2,000 training items...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Index ready: (2000, 384)  (rows=items, cols=embedding dim)


In [6]:
# retrieve_similar: find k most similar training items for a query description

def retrieve_similar(query_summary: str, k: int = 3) -> list[tuple[Item, float]]:
    """
    Embed query_summary, compute cosine similarity against index_matrix,
    return top-k (Item, score) pairs — most similar first.
    """
    q_vec = embedder.encode([query_summary], convert_to_numpy=True)
    q_vec = q_vec / (np.linalg.norm(q_vec) + 1e-9)
    scores = (index_matrix @ q_vec.T).squeeze()
    top_idx = np.argsort(scores)[::-1][:k]
    return [(index_items[i], float(scores[i])) for i in top_idx]


# Quick sanity check
sample = test[0]
similar = retrieve_similar(sample.summary, k=3)
print(f"Query    : {sample.summary[:80]}...")
print(f"Actual $ : {sample.price:.2f}\n")
for item, score in similar:
    print(f"  sim={score:.3f}  ${item.price:.2f}  {item.summary[:70]}...")


Query    : Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: O...
Actual $ : 219.00

  sim=0.594  $169.99  Title: MXR Classic 108 Fuzz Pedal  
Category: Guitar Effects Pedals  
...
  sim=0.435  $199.00  Title: Sony ECM‑VG1 Compact Shotgun Microphone  
Category: Audio Equip...
  sim=0.421  $179.00  Title: Pocket Talker Ultra System with HED 021 Deluxe Folding Headphon...


In [7]:
# Prompt builders and pricer factory

def zero_shot_messages(item: Item) -> list[dict]:
    """Minimal zero-shot prompt — same as Week 6."""
    return [
        {
            "role": "user",
            "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            ),
        }
    ]


def few_shot_messages(item: Item, k: int = 3) -> list[dict]:
    """
    RAG-augmented few-shot prompt: retrieve k analogous products,
    inject as priced examples before asking about the target item.
    """
    analogies = retrieve_similar(item.summary, k=k)
    shots = "\n\n".join(
        f"Product: {sim.summary[:220].rstrip()}...\nPrice: ${sim.price:.2f}"
        for sim, _ in analogies
    )
    return [
        {
            "role": "user",
            "content": (
                "Here are similar products with their prices:\n\n"
                f"{shots}\n\n"
                "Now estimate the price of the following product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            ),
        }
    ]


def make_pricer(client: OpenAI, model: str, prompt_fn, label: str):
    """
    Factory that returns a callable fn(item) -> str, compatible with
    pricer.evaluator.Tester.  The function's __name__ is set to label
    so Tester.make_title() renders it correctly.

    Errors are caught and printed per-item so a single failed API call
    does not abort the whole Tester run (which would prevent chart rendering).
    """
    def pricer(item: Item) -> str:
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=prompt_fn(item),
                max_tokens=12,
            )
            return resp.choices[0].message.content
        except Exception as e:
            print(f"\nError [{label}]: {type(e).__name__}: {str(e)[:160]}")
            return "0"

    pricer.__name__ = label.replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")
    pricer.__qualname__ = pricer.__name__
    return pricer


# ── Pricer registry ──────────────────────────────────────────────────────────

GROQ_MODEL = "llama-3.3-70b-versatile"
# OpenRouter model IDs use the format provider/model-name.
# If you see 404 errors below, check https://openrouter.ai/models for current IDs.
OR_MODEL   = "openai/gpt-4.1-nano"

PRICERS: dict[str, tuple] = {
    # Groq — zero-shot vs few-shot (k=3) vs few-shot (k=5)
    "Llama-3.3-70b Zero-Shot (Groq)":      (
        make_pricer(groq_client, GROQ_MODEL, zero_shot_messages,             "Llama_33_70b_ZeroShot_Groq"),
        "Groq", "zero-shot",
    ),
    "Llama-3.3-70b Few-Shot k=3 (Groq)":  (
        make_pricer(groq_client, GROQ_MODEL, lambda i: few_shot_messages(i, 3), "Llama_33_70b_FewShot_k3_Groq"),
        "Groq", "few-shot k=3",
    ),
    "Llama-3.3-70b Few-Shot k=5 (Groq)":  (
        make_pricer(groq_client, GROQ_MODEL, lambda i: few_shot_messages(i, 5), "Llama_33_70b_FewShot_k5_Groq"),
        "Groq", "few-shot k=5",
    ),
    # OpenRouter — zero-shot vs few-shot (k=3)
    "GPT-4.1-nano Zero-Shot (OR)":         (
        make_pricer(or_client,   OR_MODEL,   zero_shot_messages,             "GPT_4_1_nano_ZeroShot_OR"),
        "OpenRouter", "zero-shot",
    ),
    "GPT-4.1-nano Few-Shot k=3 (OR)":      (
        make_pricer(or_client,   OR_MODEL,   lambda i: few_shot_messages(i, 3), "GPT_4_1_nano_FewShot_k3_OR"),
        "OpenRouter", "few-shot k=3",
    ),
}

print(f"Registered {len(PRICERS)} pricers:")
for name, (fn, provider, mode) in PRICERS.items():
    print(f"  [{provider}] {name}")


Registered 5 pricers:
  [Groq] Llama-3.3-70b Zero-Shot (Groq)
  [Groq] Llama-3.3-70b Few-Shot k=3 (Groq)
  [Groq] Llama-3.3-70b Few-Shot k=5 (Groq)
  [OpenRouter] GPT-4.1-nano Zero-Shot (OR)
  [OpenRouter] GPT-4.1-nano Few-Shot k=3 (OR)


In [8]:
# Provider preflight: one test call per provider before the full benchmark.
# This surfaces model-name or auth errors immediately rather than mid-run.

def probe(client: OpenAI, model: str, label: str) -> bool:
    """Return True if the provider/model responds correctly."""
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Reply with only the number: 42"}],
            max_tokens=8,
        )
        out = resp.choices[0].message.content.strip()
        print(f"  {label}: OK  ({out!r})")
        return True
    except Exception as e:
        print(f"  {label}: FAILED — {type(e).__name__}: {str(e)[:200]}")
        return False

print("Provider preflight checks:")
groq_ok = probe(groq_client, GROQ_MODEL, f"Groq / {GROQ_MODEL}")
or_ok   = probe(or_client,   OR_MODEL,   f"OpenRouter / {OR_MODEL}")

if not or_ok:
    print("\nOpenRouter model unavailable. Switching to openai/gpt-4o-mini as fallback.")
    OR_MODEL = "openai/gpt-4o-mini"
    # Rebuild OR pricers with the fallback model
    PRICERS["GPT-4.1-nano Zero-Shot (OR)"] = (
        make_pricer(or_client, OR_MODEL, zero_shot_messages, "GPT_4o_mini_ZeroShot_OR"),
        "OpenRouter", "zero-shot",
    )
    PRICERS["GPT-4.1-nano Few-Shot k=3 (OR)"] = (
        make_pricer(or_client, OR_MODEL, lambda i: few_shot_messages(i, 3), "GPT_4o_mini_FewShot_k3_OR"),
        "OpenRouter", "few-shot k=3",
    )
    probe(or_client, OR_MODEL, f"OpenRouter / {OR_MODEL} (fallback)")

print("\nStarting benchmark...")

# Run the benchmark — Tester pass per pricer, results stored for comparison

benchmark_results: dict[str, dict] = {}

for display_name, (fn, provider, mode) in PRICERS.items():
    print(f"\n{'='*62}")
    print(f"  {display_name}")
    print(f"{'='*62}")
    try:
        t = Tester(fn, test, title=display_name, size=BENCHMARK_SIZE, workers=3)
        t.run()
        avg_err = sum(t.errors) / len(t.errors)
        r2      = r2_score(t.truths, t.guesses) * 100
        benchmark_results[display_name] = {
            "provider": provider,
            "mode": mode,
            "avg_error": avg_err,
            "r2": r2,
        }
    except Exception as exc:
        print(f"  ERROR: {exc}")
        benchmark_results[display_name] = {
            "provider": provider,
            "mode": mode,
            "avg_error": float("inf"),
            "r2": float("-inf"),
        }

print(f"\nAll {len(benchmark_results)} pricers evaluated.")



  Llama-3.3-70b Zero-Shot (Groq)


  0%|          | 0/50 [00:00<?, ?it/s]

$30 $134 $29 $50 $20 $186 $98 $95 $10 $470 $473 $129 $5 $14 $29 $7 $41 $4 $40 $31 $14 $45 $35 $75 $192 $253 $404 $7 $326 $64 $5 $40 $90 $65 $5 $19 $90 $30 $14 $12 $175 $30 $24 $5 $150 $1 $8 $13 $65 $82 


  Llama-3.3-70b Few-Shot k=3 (Groq)


  0%|          | 0/50 [00:00<?, ?it/s]

...


  GPT-4.1-nano Zero-Shot (OR)


  0%|          | 0/50 [00:00<?, ?it/s]

  ERROR: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}

  GPT-4.1-nano Few-Shot k=3 (OR)


  0%|          | 0/50 [00:00<?, ?it/s]

  ERROR: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}

All 5 pricers evaluated.


In [9]:
# Leaderboard with delta column: improvement of few-shot over zero-shot per provider

def fmt_err(v):
    return f"${v:.2f}" if v != float("inf") else "ERROR"

def fmt_r2(v):
    return f"{v:.1f}%" if v != float("-inf") else "—"


# Compute deltas per provider
provider_zero_shot: dict[str, float] = {}
for name, r in benchmark_results.items():
    if r["mode"] == "zero-shot":
        provider_zero_shot[r["provider"]] = r["avg_error"]

rows = []
for display_name, r in benchmark_results.items():
    baseline = provider_zero_shot.get(r["provider"], float("inf"))
    if r["mode"] == "zero-shot" or baseline == float("inf") or r["avg_error"] == float("inf"):
        delta_str = "—"
    else:
        delta = baseline - r["avg_error"]
        delta_str = f"{'↓' if delta > 0 else '↑'}${abs(delta):.2f}"
    rows.append({
        "Model": display_name,
        "Provider": r["provider"],
        "Mode": r["mode"],
        "Avg Err ($)": r["avg_error"],
        "R²": r["r2"],
        "Δ vs Zero-Shot": delta_str,
    })

# Sort: errors first by provider then by avg_error
rows.sort(key=lambda x: (x["Provider"], x["Avg Err ($)"]))

print(f"\n{'='*82}")
print(f"{'LEADERBOARD':^82}")
print(f"{'='*82}")
print(f"{'Model':<42} {'Provider':<12} {'Avg Err':>8} {'R²':>7} {'Δ vs Zero-Shot':>15}")
print("-" * 82)

for row in rows:
    print(
        f"{row['Model']:<42} "
        f"{row['Provider']:<12} "
        f"{fmt_err(row['Avg Err ($)']):>8} "
        f"{fmt_r2(row['R²']):>7} "
        f"{row['Δ vs Zero-Shot']:>15}"
    )

print("-" * 82)

# Summary
groq_zero = provider_zero_shot.get("Groq", None)
groq_best = min(
    (r["avg_error"] for r in benchmark_results.values() if r["provider"] == "Groq"),
    default=None,
)
if groq_zero and groq_best and groq_zero != float("inf"):
    improvement_pct = (groq_zero - groq_best) / groq_zero * 100
    print(f"\nBest Groq model reduced zero-shot error by {improvement_pct:.1f}%")
    print("This gap quantifies how much in-context retrieval contributes vs. fine-tuning.")



                                   LEADERBOARD                                    
Model                                      Provider      Avg Err      R²  Δ vs Zero-Shot
----------------------------------------------------------------------------------
Llama-3.3-70b Few-Shot k=5 (Groq)          Groq           $73.03   52.2%         ↓$11.64
Llama-3.3-70b Few-Shot k=3 (Groq)          Groq           $80.74   43.5%          ↓$3.93
Llama-3.3-70b Zero-Shot (Groq)             Groq           $84.67   27.1%               —
GPT-4.1-nano Zero-Shot (OR)                OpenRouter      ERROR       —               —
GPT-4.1-nano Few-Shot k=3 (OR)             OpenRouter      ERROR       —               —
----------------------------------------------------------------------------------

Best Groq model reduced zero-shot error by 13.8%
This gap quantifies how much in-context retrieval contributes vs. fine-tuning.


## Key Findings

### Why few-shot RAG works
The LLM already knows *how to price things* — it just lacks grounding for niche or
unusual products.  By supplying k similar items with known prices, we give the model
an implicit price distribution for that segment.  It no longer has to rely purely on
parametric memory.

### Why fine-tuning is still better (Week 7's lesson)
- Few-shot RAG adds latency (embedding + retrieval per request) and increases prompt length (cost).
- The fine-tuned Llama (QLoRA, Week 7) internalises this retrieval knowledge into weights.
  At inference it sees only the product description — no examples needed — but still achieves
  lower average error because the pricing patterns are baked in.
- Fine-tuning is the asymptote; RAG-augmentation is a strong no-training approximation.

### Progression across weeks

| Approach | Week | Avg Error (indicative) | GPU required? |
|----------|------|----------------------|---------------|
| Zero-shot LLM | 6 | highest | No |
| RAG few-shot (this notebook) | 7 | lower | No |
| QLoRA fine-tuned Llama-3.2-3B | 7 | lowest | Yes (A100) |
| Human baseline | 6 | ~$87 | No |


In [ ]:
# Week 7 fine-tuning sketch — informational only (requires A100 GPU on Colab)
# This mirrors the config in NEW_Week_7_Day_3_TRAINING.ipynb

QLORA_CONFIG_SKETCH = """
# ── QLoRA Fine-Tuning (Week 7 Day 3, runs on Google Colab A100) ──

BASE_MODEL  = "meta-llama/Llama-3.2-3B"
DATASET     = "ed-donner/items_prompts_full"   # 800 k price-annotated products

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

lora_config = LoraConfig(
    r=256,                          # high rank for large dataset
    lora_alpha=512,
    lora_dropout=0.1,
    target_modules=ATTENTION_LAYERS + MLP_LAYERS,  # all linear layers
    task_type="CAUSAL_LM",
)

# After 3 epochs, the model predicts prices from description alone:
# Input:  "What does this cost?\\n\\n<product summary>\\n\\nPrice is $"
# Output: "49.00"
#
# ed-donner's fine-tuned checkpoint achieves ~$62 avg error on the test set,
# beating human performance ($87) and our best zero-shot LLM.
# Our few-shot RAG pricer closes part of this gap — no training needed.
"""

print(QLORA_CONFIG_SKETCH)
print("To run the full fine-tuning, open:")
print("  week7/NEW_Week_7_Day_3_TRAINING.ipynb  (requires T4/A100 on Google Colab)")
print("\nTo evaluate the pre-trained checkpoint, open:")
print("  week7/NEW_Week_7_Day_5_Testing_our_Fine_tuned_model.ipynb")
